# Getting Started

In this notebook, we'll walk through a usage example of the library. In the first block, we'll use the higher-level Python wrapper. In the second block, we'll use the lower-level Cython extension directly.

We'll use fake generated data that is stored in the repository under `src/assets/fake_options_data.parquet`.

## OpenMP
The implementation depends on the usage of `OpenMP` to parallelize computations. Apple's system clang ships without OpenMP support, so we have to explicitly pull in Homebrew's LLVM and libomp. This is done in the `CMakeLists.txt` file.

To check the maximum available threads, and set the number of threads yourself, should you need more control, you can import such utility functions from `src.ext.omp_utils`.

In [ ]:
from pathlib import Path

import numpy as np
import polars as pl

from src.ext.trapezoid_rnm import compute_trapz_rnm, OPT_CALL, OPT_PUT
from src.ext.omp_utils import get_max_threads, set_num_threads
from src.risk_neutral_moments import DataSchema, risk_neutral_moments

In [4]:
print(f"Maximum number of threads available: {get_max_threads()}")

Maximum number of threads available: 8


In [5]:
set_num_threads(8)

## High-Level Python Wrapper
The Python wrapper depends on `Polars`. An extension of the function is the dataclass `DataSchema`. Here, you can pass column names as well as the representations of how call and put options are identified. The data is epxected to be in long format. See the docstring for more information.
The output of the function is a `Polars` data frame with the following columns:
- stock_id
- date
- varQ - variance under measure $\mathbb{Q}$
- volQ - annualized volatilty under measure $\mathbb{Q}$
- skewQ - skewness under measure $\mathbb{Q}$
- kurtQ - kurtosis under measure $\mathbb{Q}$

In [6]:
help(DataSchema)

Help on class DataSchema in module src.risk_neutral_moments:

class DataSchema(builtins.object)
 |  DataSchema(
 |      stock_identifier: str = 'stock_id',
 |      date: str = 'date',
 |      option_type: str = 'option_type',
 |      call_flag: str = 'call',
 |      put_flag: str = 'put',
 |      spot_price: str = 'spot_price',
 |      strike_price: str = 'strike_price',
 |      time_to_maturity: str = 'time_to_maturity',
 |      implied_volatility: str = 'implied_volatility',
 |      rf_rate: str = 'rf_rate'
 |  ) -> None
 |
 |  A class to hold the column names for the input data
 |
 |  Parameters
 |  ----------
 |  stock_identifier : str
 |      Key identifier for the stocks, e.g., the CRSP PERMNO.
 |      By default, "stock_id" is used as the column name for this field.
 |  date : str
 |      The date of the observation. By default, "date" is used as the column
 |      name for this field.
 |  option_type : str
 |      Call / Put flag identifier column name. By default, "option_type

In [2]:
path = Path.cwd() / "src" / "assets" / "fake_options_data.parquet"

df = pl.read_parquet(path).with_columns(pl.lit("A").alias("stock_id"))
df

date,expiry,days,T,risk_free_rate,strike,option_type,bid,ask,mid,implied_vol,open_interest,volume,spot,stock_id
date,date,i32,f64,f64,f64,str,f64,f64,f64,f64,i32,i32,f64,str
2026-03-20,2026-03-27,7,0.019178,0.043,4325.0,"""put""",0.05,0.05,0.05,0.182101,130,36,5764.03,"""A"""
2026-03-20,2026-03-27,7,0.019178,0.043,4390.0,"""put""",0.05,0.05,0.05,0.185015,74,15,5764.03,"""A"""
2026-03-20,2026-03-27,7,0.019178,0.043,4460.0,"""put""",0.05,0.05,0.05,0.183018,551,175,5764.03,"""A"""
2026-03-20,2026-03-27,7,0.019178,0.043,4525.0,"""put""",0.05,0.05,0.05,0.17203,413,112,5764.03,"""A"""
2026-03-20,2026-03-27,7,0.019178,0.043,4595.0,"""put""",0.05,0.05,0.05,0.171558,468,114,5764.03,"""A"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-04-02,2027-03-19,351,0.961644,0.040576,7135.0,"""call""",81.28,84.66,82.97,0.152513,609,103,5929.6,"""A"""
2026-04-02,2027-03-19,351,0.961644,0.040576,7205.0,"""call""",65.52,68.28,66.9,0.14783,52,11,5929.6,"""A"""
2026-04-02,2027-03-19,351,0.961644,0.040576,7270.0,"""call""",61.34,63.96,62.65,0.150246,953,107,5929.6,"""A"""


In [ ]:
# Store the column names and representations of call and put options in
# DataSchema. The Polars function uses this class to extract and transform the
# required information and consequently pass to the Cython wrapper.
ds = DataSchema(
    stock_identifier="stock_id",
    date="date",
    option_type="option_type",
    call_flag="call",
    put_flag="put",
    spot_price="spot",
    strike_price="strike",
    time_to_maturity="days",
    implied_volatility="implied_vol",
    rf_rate="risk_free_rate",
)
res = risk_neutral_moments(options_data=df, data_schema=ds, group_freq="1d")
res

stock_id,date,varQ,volQ,skewQ,kurtQ
str,date,f64,f64,f64,f64
"""A""",2026-03-20,0.000666,0.089404,-1.538269,2.928777
"""A""",2026-03-23,0.000381,0.067625,-1.535386,2.866963
"""A""",2026-03-24,0.000289,0.058866,-1.543385,2.840854
"""A""",2026-03-25,0.000191,0.047832,-1.615143,3.001515
"""A""",2026-03-26,0.000093,0.033425,-1.695738,3.135663
"""A""",2026-03-27,0.000665,0.089338,-1.522623,2.860263
"""A""",2026-03-30,0.000374,0.067026,-1.560475,2.902077
"""A""",2026-03-31,0.000273,0.057266,-1.62887,3.030369
"""A""",2026-04-01,0.000187,0.047362,-1.610554,2.932711


## Use Cython Extension Directly
The user can also opt for using the Cython extension directly. The Cython function expects the `indptr` array to be an array of CSR-style row pointers. We create some fake testing data by hand. There are a total of 2 trading days, and each day has 5 OTM call and put options. From these OTM options, we compute the risk neutral moments for each group.

The function returns 4 arrays:
- Variance under measure $\mathbb{Q}$
- Skewness under measure $\mathbb{Q}$
- Kurtosis under measure $\mathbb{Q}$
- Return code, with 0 meaning success

In [9]:
help(compute_trapz_rnm)

Help on cython_function_or_method in module src.ext.trapezoid_rnm:

compute_trapz_rnm(strikes, ivols, flags, spots, rf, ttm, indptr)
    Compute risk-neutral moments for all groups in parallel.

    Bridge between the Python API and the C helper functions. This function does the
    following:
    - Accept pre-processed flat NumPy arrays + CSR-style group index pointers.
    - Validate input consistency and data availability.
    - Iterate over groups in parallel with OpenMP prange.
    - Call C-code under nogil for each group.
    - Write scalar results into pre-allocated output arrays; return as NumPy arrays

    Parameters
    ----------
    strikes : ndarray, shape (N,)
        OTM strike prices across all groups.
    ivols : ndarray, shape (N,)
        Corresponding implied volatilities.
    flags : ndarray, shape (N,), dtype int
        Option type: OPT_CALL (1) or OPT_PUT (0).
    spots : ndarray, shape (G,)
        Spot price for each group.
    rf : ndarray, shape (G,)
       

In [ ]:
# Day 1: 5 OTM calls, 5 OTM puts
# Day 2: 5 OTM calls, 5 OTM puts
strikes = np.array([
    # Day 1: calls
    105.0, 110.0, 115.0, 120.0, 125.0,
    # Day 1: puts
    95.0, 90.0, 85.0, 80.0, 75.0,
    # Day 2: calls
    210.0, 220.0, 230.0, 240.0, 250.0,
    # Day 2: puts
    190.0, 180.0, 170.0, 160.0, 150.0,
], dtype=np.float64)

ivols = np.array([
    # Day 1 calls and puts
    0.20, 0.22, 0.25, 0.28, 0.30,
    0.30, 0.28, 0.25, 0.22, 0.20,
    # Day 2 calls and puts
    0.18, 0.20, 0.23, 0.26, 0.28,
    0.28, 0.26, 0.23, 0.20, 0.18,
], dtype=np.float64)

flags = np.array([
    # Day 1: 5 calls and puts
    OPT_CALL, OPT_CALL, OPT_CALL, OPT_CALL, OPT_CALL,
    OPT_PUT, OPT_PUT, OPT_PUT, OPT_PUT, OPT_PUT,
    # Day 2: 5 calls and puts
    OPT_CALL, OPT_CALL, OPT_CALL, OPT_CALL, OPT_CALL,
    OPT_PUT, OPT_PUT, OPT_PUT, OPT_PUT, OPT_PUT,
], dtype=np.int32)

# parameters per trading day (e.g., this may be end of day data)
spots = np.array([100.0, 200.0], dtype=np.float64)
rf = np.array([0.05, 0.05], dtype=np.float64)
ttm = np.array([0.25, 0.25], dtype=np.float64)

# CSR-style indices: Day 1 and 2 both have 10 options ('group' sizes)
sizes = np.array([10, 10])
indptr = np.zeros(len(sizes) + 1, dtype=np.intp)
np.cumsum(sizes, out=indptr[1:])
# In this case we could also simply do:
# indptr = np.array([0, 10, 20], dtype=np.intp)


var, skew, kurt, rc = compute_trapz_rnm(
    strikes,
    ivols,
    flags,
    spots,
    rf,
    ttm,
    indptr
)

print("Day 1:")
print(f"  Variance: {var[0]:.4f}")
print(f"  Skewness: {skew[0]:.4f}")
print(f"  Kurtosis: {kurt[0]:.4f}")
print(f"  Return Code: {rc[0]}\n")

print("Day 2:")
print(f"  Variance: {var[1]:.4f}")
print(f"  Skewness: {skew[1]:.4f}")
print(f"  Kurtosis: {kurt[1]:.4f}")
print(f"  Return Code: {rc[1]}")

Day 1:
  Variance: 0.0142
  Skewness: -1.8842
  Kurtosis: 3.2031
  Return Code: 0

Day 2:
  Variance: 0.0119
  Skewness: -1.9490
  Kurtosis: 3.4347
  Return Code: 0
